# Content Classification Basic Tutorial

## Overview

In this 10-15 min tutorial we will cover

> - How to get a category classification for your document / text using either  
        - Global categories that exist as a boiler plate for all tenants
        - Your own categories of interest defined by uploading file or text examples
> - How to set up and query your own specific categories.
> - Using custom or 'out of the box' tags to enrich the metadata we have for a file. 

Supported file types:
>- PDF: pdf
>- Word: docx, docm, doc
>- Excel: xlsx, xlsm, xls, xltx, xltm
>- PowerPoint: pptx, pptm, ppt, pps, potx, potm, pot, ppsx, ppsm
>- Eml : eml
>- Text : txt, cs, cpp, go, java, js, php, py, rs, swift (excluding tags)

### Imports

In [1]:
import os
import sys
import time
from pathlib import Path
sys.path.append(os.path.abspath("../src"))

from content_classification_client_simple import ContentClassificationClient

/Users/aviavidan/envs/py39/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


### Initialize a Content Classification Client

In the code block below we initialize the client with your auth key. make sure to **set it up as an ENV VAR before launching jupyter notebook** in the terminal.

In [2]:
AUTH_KEY = os.environ['AUTH_KEY']

client = ContentClassificationClient(
    base_url="https://content-classification-prod.us.paralus.votiro.com/",
    auth_token=AUTH_KEY
)

### Inference example - code files classification

In the code block below we will use *client.categorize_file(file_path)* to POST files to this endpoint */inference/file* and print out the results

In [3]:
file_folder = "../../query_files"
folder = Path(file_folder) / "code" 

for root, dirs, files in os.walk(folder):

    for i, file in enumerate(files):
        if file.startswith('.DS'):
            continue
        full_path = os.path.join(root, file)
        print(f'#{i+1}, {file.split("_")[0]} code snippet')
        print(client.categorize_file(full_path))

#1, linked-list.cpp code snippet
{'results': [{'category': 'code', 'is_global': True, 'confidence': '0.8354018386953358', 'category_file_id': 'Ora8tJkB7zJ0hG38Lts5'}]}
#3, form-required.php code snippet
{'results': [{'category': 'code', 'is_global': True, 'confidence': '0.7719006405696178', 'category_file_id': 'Sra8tJkB7zJ0hG38T9tb'}]}
#4, marquee.cs code snippet
{'results': [{'category': 'code', 'is_global': True, 'confidence': '0.8351070395196305', 'category_file_id': 'O7a8tJkB7zJ0hG38MNu3'}]}
#5, Ispalindrome.py code snippet
{'results': [{'category': 'code', 'is_global': True, 'confidence': '0.8226324431175644', 'category_file_id': 'Q7a8tJkB7zJ0hG38QNsS'}]}
#6, callback.js code snippet
{'results': [{'category': 'code', 'is_global': True, 'confidence': '0.8391912682478382', 'category_file_id': 'Sra8tJkB7zJ0hG38T9tb'}]}
#7, bubble-sort.rs code snippet
{'results': [{'category': 'code', 'is_global': True, 'confidence': '0.8716149114704728', 'category_file_id': 'O7a8tJkB7zJ0hG38MNu3'}]}


### Understanding Inference Results

For all code snippents above (that are located under ../../query_files/code), we got the category predicted as 'code' with confidence scores bw 0.8-0.9. 

**IMPORTANT**: a confidence score around and below 0.75 would be considered low and this category should NOT be considered a good match for the file / text sample. a result with confidence score below 0.75 will not be returned.

We also get back these fields
>- *is_global (bool)* - True means this is a boiler plate category we offer all tenants, mostly for demonstration purposes. below we will review an example to setup your own relevant categories - these will be specific for your tenant. below we will also show how to include some or none of these global categories when inferencing.
>- *category_file_id (str)* - this field will be usefull when you wish to erase any of your tenant specific category_file.

category_file is the name we give for a file (or text example) that represents a category. below you will learn how to add / remove category_file to represent new and existing categories.

*NOTICE* that many files could represent the same category, since it is unlikely that a signle file will encapsulate all semantics associated with a category.

*Also IMPORTANT to notice* that same file (or text example) could point to different categories, but this is Un-recommended and should be avoided.

### Inference example - text classification

Here we used client.categorize_text(some_text) to POST a text to this endpoint "/inference/text". 

Results for "/inference/file" are in the same format as for text.

In [4]:
client.categorize_text("[int(x.strip(" "), 10) for x in list_str.split(',')]")

{'results': [{'category': 'code',
   'is_global': True,
   'confidence': '0.8392541232233988',
   'category_file_id': 'PLa8tJkB7zJ0hG38M9s-'}]}

*NOTICE*: short text or code snippets might not get the proper classification due to weak / inconclusive semantics 

### Inference example - text classification with various results 

In [5]:
client.categorize_text(["a strategic planning tool used to identify an organization’s Strengths, Weaknesses, Opportunities, and Threats"])

{'results': [{'category': 'swot_analysis',
   'is_global': True,
   'confidence': '0.8002233708460038',
   'category_file_id': 'E1k3T5cBmIvvcEA8aI8X'},
  {'category': 'employee_performance',
   'is_global': True,
   'confidence': '0.7810656784839517',
   'category_file_id': 'Blk3T5cBmIvvcEA8Z48u'},
  {'category': 'employee_demographic',
   'is_global': True,
   'confidence': '0.7715063236756914',
   'category_file_id': 'A1k3T5cBmIvvcEA8Zo_0'}]}

For the above 'swot_analysis' definition (posted as plain text) we got more than one result, *sorted by confidence* in descending order. 

>THE HIGHER THE CONFIDENCE THE MORE MEANINGFUL THE CATEGORY IS.  
>INFERENCE RESULTS WILL ALWAYS BE SORTED WITH HIGH CONFIDENCE AT THE TOP.

You can control the number of results by passing a parameter: number_of_results = 1

In [6]:
client.categorize_text(
    "a strategic planning tool used to identify an organization’s Strengths, Weaknesses, Opportunities, and Threats",
    number_of_results=1)

{'results': [{'category': 'swot_analysis',
   'is_global': True,
   'confidence': '0.8002233708460038',
   'category_file_id': 'E1k3T5cBmIvvcEA8aI8X'}]}

### 'Out of The Box' Global Categories

Global categories are preconfigured by the platform and shared across tenants.

Let's GET the available categories using the */management/get_categories* endpoint, by calling *client.get_categories()*

In [7]:
client.get_categories(approved_only_filter=True)

{'categories': [{'category_name': 'HR_training', 'is_global': True},
  {'category_name': 'business_data', 'is_global': True},
  {'category_name': 'code', 'is_global': True},
  {'category_name': 'cv', 'is_global': True},
  {'category_name': 'employee_compensation', 'is_global': True},
  {'category_name': 'employee_demographic', 'is_global': True},
  {'category_name': 'employee_performance', 'is_global': True},
  {'category_name': 'legal_data', 'is_global': True},
  {'category_name': 'payslip', 'is_global': True},
  {'category_name': 'safety_training', 'is_global': True},
  {'category_name': 'swot_analysis', 'is_global': True}]}

**Global Categories** are pre-defined and will be marked with the flag 'is_global' == True.

These are meant mostly for educational purposes but you may find all / some of them useful to your usecases.

### Inference with Some of the Categories

Using the *global_categories_filter* we can control exactly what global categories we wish to include in our query.
>- You can pass '' empty string to not include any of the global categories
>- You can pass '*' to not include all of the global categories - this is the DEFAULT behaviour
>- You can include specific global categories like in the code block below by setting *global_categories_filter='swot_analysis,code'*

In the same manner, using *tenant_categories_filter* you can control which tenant specific categories will be queried.

In [8]:
client.categorize_text(
    "a strategic planning tool used to identify an organization’s Strengths, Weaknesses, Opportunities, and Threats",
    global_categories_filter='swot_analysis,code')

{'results': [{'category': 'swot_analysis',
   'is_global': True,
   'confidence': '0.8002233708460038',
   'category_file_id': 'E1k3T5cBmIvvcEA8aI8X'}]}

### Setting up a New Category

In this section we will setup a new category to identify documents describing synthetic data generation, similar to these two files below.

In [9]:
folder = Path(file_folder) / "synthetic_data_gen" 

file_paths = []
for root, dirs, files in os.walk(folder):
    for i, file in enumerate(files):
        full_path = os.path.join(root, file)
        file_paths.append(full_path)
        print(f'#{i+1}, {file}')
        results = client.categorize_file(full_path)
        print(results)

#1, Synthetic_Data_Generation_Implementation_Guide.pdf
{'results': []}
#2, Synthetic_Data_Generation_Overview.pdf
{'results': []}


None of the global categories is relevant for these documents **so we get no results back**.

Lets preview the documents using *client.preview_pdf(file_path)*

In [10]:
print(client.preview_pdf(file_paths[0])) # preview 1st file

Synthetic Data Generation - Technical
Implementation Guide
 
Introduction
This document outlines the technical implementation of synthetic data generation workflows for
machine learning practitioners. The goal is to enable scalable, reproducible, and privacy-compliant
synthetic data pipelines.
Pipeline Design
A robust synthetic data pipeline typically includes the following stages: 1. Data Profiling: Analyze
distributions, correlations, and feature importance. 2. Model Selection: Choose between GANs,
VAEs, or tabular synthesis models like CTGAN. 3. Generation: Train the model using anonymized
or sample data. 4. Validation: Compare real vs synthetic distributions using statistical metrics such
as KS-test, Chi-square, or Wasserstein distance. 5. Deployment: Serve synthetic data through APIs
or data catalogs.
Tools and Frameworks
Several open-source tools support synthetic data generation: - SDV (Synthetic Data Vault):
Supports relational, sequential, and tabular data synthesis. - Gretel.

In [11]:
print(client.preview_pdf(file_paths[1])) # preview 2nd file

Synthetic Data Generation for Machine Learning -
 Overview
 
Introduction
Synthetic data generation refers to the process of creating artificial datasets that mimic real-world
data. These datasets are often used in machine learning to address privacy concerns, data
scarcity, or bias issues. Synthetic data can replicate the statistical properties of real data while
avoiding direct exposure of sensitive information.
Techniques for Synthetic Data Generation
1. Statistical Sampling: Uses probability distributions derived from real data to generate new
samples. 2. Generative Models: Includes GANs (Generative Adversarial Networks) and VAEs
(Variational Autoencoders) that learn complex data structures. 3. Rule-based Simulation: Domain
experts define logical rules and constraints to simulate data. 4. Agent-based Modeling: Used in
environments like finance or traffic to simulate entity behaviors.
Applications in Machine Learning
Synthetic data is especially valuable when dealing with limited la

### Automated Category Discovery

Upon completion of full integration and with files flowing in, we employ our proprietary solution to **automatically identify and suggest relevant categories** based on each tenant's data distribution to maximize the quality of the detection and to reduce both False Positives and False Negatives.

While this proprietary process is not part of this tutorial, below we will see how easy it is to add new categories.

### Setup or Extend Your Own Categories by Adding a Category-File

Using *client.add_category_file()* we POST one of these files to */management/add_category_file*, basically using it to represent **a new category called 'synthetic_data_gen'**. 

*NOTICE*: the name of the category could be whatever you want it to be. changing the name will not impact the classification results (only adding or removing category_files will).

In [12]:
client.add_category_file(category_name="synthetic_data_gen", file=file_paths[0], is_approved=True)

{'message': "Category file for 'synthetic_data_gen' category file of tenant 30b95dc8-c157-4eba-a2ce-67379d07dc12 added successfully."}

In [13]:
time.sleep(5) # wait to make sure update takes effect

In [13]:
category_file_ids = [] # we collect these to later remove the files added for demonstration purposes
for i, full_path in enumerate(file_paths):
    print(f'#{i+1}, {file}')
    results = client.categorize_file(full_path)
    print(results) 
    category_file_ids.extend(r.get('category_file_id') for r in results.get('results', []))

#1, Synthetic_Data_Generation_Overview.pdf
{'results': [{'category': 'synthetic_data_gen', 'is_global': False, 'confidence': '1.0000000000000009', 'category_file_id': 'EgrulZoB7zJ0hG38_33u'}]}
#2, Synthetic_Data_Generation_Overview.pdf
{'results': [{'category': 'synthetic_data_gen', 'is_global': False, 'confidence': '0.8442753811911139', 'category_file_id': 'EgrulZoB7zJ0hG38_33u'}]}


Now, immediately after adding this new category, **both files are classified properly as 'synthetic_data_gen'**  

*ADVANCED TIP*: adding a category_file with *is_approved=False* will allow you to control / evaluate the classification results with and without specific category_files.

#### List All Categories: Global & Tenant Specific

Now, you can see your new category with the flag *is_global == False* (since it is only specific to your tenant and not visible to any other tenant).

In [14]:
client.get_categories(approved_only_filter=True)

{'categories': [{'category_name': 'HR_training', 'is_global': True},
  {'category_name': 'business_data', 'is_global': True},
  {'category_name': 'code', 'is_global': True},
  {'category_name': 'cv', 'is_global': True},
  {'category_name': 'employee_compensation', 'is_global': True},
  {'category_name': 'employee_demographic', 'is_global': True},
  {'category_name': 'employee_performance', 'is_global': True},
  {'category_name': 'legal_data', 'is_global': True},
  {'category_name': 'payslip', 'is_global': True},
  {'category_name': 'safety_training', 'is_global': True},
  {'category_name': 'swot_analysis', 'is_global': True},
  {'category_name': 'synthetic_data_gen', 'is_global': False}]}

In [16]:
### Remove Added Category-Files
[client.delete_category_file(file_id) for file_id in set(category_file_ids)]

[{'message': "Tenant's 30b95dc8-c157-4eba-a2ce-67379d07dc12 category file with id='_grtlZoB7zJ0hG38hXxA' deleted successfully."}]

### Extracting tags of interest (custom tags) - 1st example

You can query any file with a list of your own custom tags comma delimited (no spaces).

We will now POST these file to */tags/get_tags* using *client.extract_tags()*

In [17]:
custom_tags = 'machine learning,agentic,augmented nlp'

for i, full_path in enumerate(file_paths):
    print(f'#{i+1}, {file}')
    print(client.extract_tags(full_path, custom_tags=custom_tags))

#1, Synthetic_Data_Generation_Overview.pdf
{'tags': [{'tag': 'machine learning', 'confidence': 0.934}]}
#2, Synthetic_Data_Generation_Overview.pdf
{'tags': [{'tag': 'agentic', 'confidence': 0.996}, {'tag': 'augmented nlp', 'confidence': 0.95}, {'tag': 'machine learning', 'confidence': 0.928}]}


Again each relevant tag will get a confidence score. results are sorted by confidence.

Reviewing the results above, we can see that while both documents refer to 'machine learning', the second also refers to 'agentic' and 'augmented nlp'

### Extracting tags of interest (custom tags) - 2nd example

In [18]:
custom_tags = 'performance bonus,incentive plan,retirement savings'

client.extract_tags(Path(file_folder) / "tags" / 'example_01.pdf', custom_tags=custom_tags)

{'tags': [{'tag': 'retirement savings', 'confidence': 0.965},
  {'tag': 'performance bonus', 'confidence': 0.851},
  {'tag': 'incentive plan', 'confidence': 0.763}]}

### 'Out of the Box' Tags - a few examples

In [19]:
file_path = Path(file_folder) / "tags" / "Invoice_Example_NonTech.pdf"
client.extract_tags(file_path)

{'tags': [{'tag': 'Financial data', 'confidence': 0.791}]}

In [20]:
file_path = Path(file_folder) / "tags" / "HR_Policies_and_Standards.pdf"
client.extract_tags(file_path)

{'tags': [{'tag': 'Performance data/review', 'confidence': 0.961},
  {'tag': 'HR', 'confidence': 0.956},
  {'tag': 'Training', 'confidence': 0.945},
  {'tag': 'Absenteeism', 'confidence': 0.931},
  {'tag': 'Compensation', 'confidence': 0.885}]}

In [21]:
file_path = Path(file_folder) / "tags" / "Invoice_Example.pdf"
client.extract_tags(file_path)

{'tags': [{'tag': 'Training', 'confidence': 0.881},
  {'tag': 'Financial data', 'confidence': 0.753},
  {'tag': 'Operational data production', 'confidence': 0.728}]}

### Difference Between Classification and Tags

**IMPORTANT**: the detection of a tag is softer compared to to the content classification shown in the first part of this tutorial.  
The presence of a tag means this term or concept was mentioned but does not at all guarantee the higher level categorization of the document or the  text. Therefore, tags could be used to complement the classification result.

To sum up: tags capture surface-level mentions of concepts, while classifications indicate document-level relevance.

## Summary

In this short tutorial we covered 

> - How to get a category classification for your pdf document / text if it is related to one of the global categories that exist as a boiler plate for all tenants
> - How to set up and query your own specific categories (using a file example or a text example). In the case that a document is not classified properly to it's category you should consider adding it as another example to improve the representation of that category.
> - Using custom or 'out of the box' tags to enrich the metadata we have for a file. for example, this mechanism could be used to distinguish if an HR document is a general one, or one of a specific company (cause it has that company name identified as a tag, etc). 